# 📚 RAG Application

## Retrieval-Augmented Generation for PDF Question Answering

This notebook demonstrates a Retrieval-Augmented Generation (RAG) pipeline that allows users to upload a PDF, extract and chunk its text, store the document in ChromaDB, retrieve relevant context, and generate answers using Groq.

In [1]:
import os
from io import BytesIO
import chromadb
from dotenv import load_dotenv
from groq import Groq
from pypdf import PdfReader

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file")

groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq API key loaded successfully!")

Groq API key loaded successfully!


In [3]:
GROQ_MODEL = "llama-3.1-8b-instant"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 4

print("Configuration loaded successfully!")

Configuration loaded successfully!


In [4]:
pdf_path = input("Enter the full path to your PDF: ")

print("PDF selected:")
print(pdf_path)

PDF selected:
C:/Users/kiran/Downloads/AI_resume.pdf


In [5]:
import os

if os.path.exists(pdf_path):
    print("PDF found successfully!")
else:
    print("PDF not found. Please check the path.")

PDF found successfully!


In [6]:
reader = PdfReader(pdf_path)

pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text()

    if text and text.strip():
        pages.append({
            "page": page_number,
            "text": text.strip()
        })

print(f"Pages extracted: {len(pages)}")

Pages extracted: 1


In [7]:
print(pages[0]["text"][:2000])

Usarthi Kiran Babu 
+91-9014005959 | kiranbabuusarthi2@gmail.com | Guntur, Andhra Pradesh 
GitHub:https://github.com/KiranBabu2003 | LeetCode : https://leetcode.com/u/usarthi_kiranbabu/ 
LinkedIn : https://www.linkedin.com/in/kiran-babu-usarthi-1bba142a0/ | 
Career Objective 
Aspiring AI/ML Engineer with hands -on experience in machine learning, deep learning, and AI -powered web 
applications. Skilled in Python, Scikit -learn, and TensorFlow/PyTorch. Passionate about Generative AI, LLM 
applications, and building intelligent systems that solve real-world problems. 
 Education  
 
Degree/Certificate Institute/Board CGPA/Percentage Year 
B. Tech in CSE SRKR Engineering College, Bhimavaram 8.41 2022-2026 
Intermediate (MPC) Bhashyam Junior College, Guntur 91.4% 2022 
SSC Bhashyam High School, Guntur 97.5% 2020 
Internship 
NIELIT | AI / ML – Intern | June 2024 - July 2024 
 Built a heart attack prediction system (85%+ accuracy) using ML with full-stack integration.
 Improved model per

In [8]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

chunks = []

for page in pages:
    text = page["text"]
    page_number = page["page"]

    start = 0

    while start < len(text):
        end = start + CHUNK_SIZE

        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "text": chunk_text,
                "page": page_number
            })

        start += CHUNK_SIZE - CHUNK_OVERLAP

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 4


In [9]:
print("First chunk:")
print(chunks[0]["text"])
print()
print("Page:", chunks[0]["page"])

First chunk:
Usarthi Kiran Babu 
+91-9014005959 | kiranbabuusarthi2@gmail.com | Guntur, Andhra Pradesh 
GitHub:https://github.com/KiranBabu2003 | LeetCode : https://leetcode.com/u/usarthi_kiranbabu/ 
LinkedIn : https://www.linkedin.com/in/kiran-babu-usarthi-1bba142a0/ | 
Career Objective 
Aspiring AI/ML Engineer with hands -on experience in machine learning, deep learning, and AI -powered web 
applications. Skilled in Python, Scikit -learn, and TensorFlow/PyTorch. Passionate about Generative AI, LLM 
applications, and building intelligent systems that solve real-world problems. 
 Education  
 
Degree/Certificate Institute/Board CGPA/Percentage Year 
B. Tech in CSE SRKR Engineering College, Bhimavaram 8.41 2022-2026 
Intermediate (MPC) Bhashyam Junior College, Guntur 91.4% 2022 
SSC Bhashyam High School, Guntur 97.5% 2020 
Internship 
NIELIT | AI / ML – Intern | June 2024 - July 2024 
 Built a heart attack prediction system (85%+ accuracy) using ML with full-stack integration.
 Impro

In [10]:
chroma_client = chromadb.PersistentClient(
    path="chroma_db"
)

collection = chroma_client.get_or_create_collection(
    name="rag_notebook_documents"
)

print("ChromaDB collection ready!")

ChromaDB collection ready!


In [11]:
documents = [chunk["text"] for chunk in chunks]

ids = [
    f"chunk_{index}"
    for index in range(len(chunks))
]

metadatas = [
    {
        "page": chunk["page"],
        "source": os.path.basename(pdf_path)
    }
    for chunk in chunks
]

collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas
)

print(f"Stored {len(chunks)} chunks in ChromaDB.")

Stored 4 chunks in ChromaDB.


In [12]:
question = input("Ask a question about your PDF: ")

print("Question:", question)


Question: What programming languages are mentioned in the resume?


In [13]:
results = collection.query(
    query_texts=[question],
    n_results=4
)

retrieved_documents = results["documents"][0]
retrieved_metadatas = results["metadatas"][0]

print(f"Retrieved {len(retrieved_documents)} relevant chunks.")

for index, (document, metadata) in enumerate(
    zip(retrieved_documents, retrieved_metadatas),
    start=1
):
    print(f"\n--- Source {index} | Page {metadata['page']} ---")
    print(document)

Retrieved 4 relevant chunks.

--- Source 1 | Page 1 ---
s.
 Evaluated model performance using accuracy and validation techniques. 
OCT Retinal Disease Classification System 
Tools: Python, TensorFlow/PyTorch, CNN Github 
• Developed a deep learning model to classify OCT retinal images into four disease categories. 
• Applied image preprocessing and CNN techniques to improve prediction accuracy and enable real-
time prediction. 
 Skills  
 Programming and Data: C, Python, Java, SQL
 AI/ML: Scikit-learn, Regression, Classification, Random Forest, Feature Engineering, Model Training & 
Evaluation
 Deep Learning: CNN, RessNet, MobileNet, ViT
 Generative AI: Prompt Engineering, RAG basics, LLM workflows, OpenAI APIs, LangChain (Basics) 
 Web Technologies: HTML, CSS, JavaScript (basics), Flask/FastAPI
 Tools: Git, GitHub
 Certificates  
ML Using Python – NIELIT | Web Technologies – SRKR College | Python (Basic) – HackerRank 
Programming in C++ – Cisco | Machine Learning – MLConn

In [14]:
context_parts = []

for index, (document, metadata) in enumerate(
    zip(retrieved_documents, retrieved_metadatas),
    start=1
):
    context_parts.append(
        f"Source {index} | Page {metadata['page']}\n{document}"
    )

context = "\n\n".join(context_parts)

print(context)

Source 1 | Page 1
s.
 Evaluated model performance using accuracy and validation techniques. 
OCT Retinal Disease Classification System 
Tools: Python, TensorFlow/PyTorch, CNN Github 
• Developed a deep learning model to classify OCT retinal images into four disease categories. 
• Applied image preprocessing and CNN techniques to improve prediction accuracy and enable real-
time prediction. 
 Skills  
 Programming and Data: C, Python, Java, SQL
 AI/ML: Scikit-learn, Regression, Classification, Random Forest, Feature Engineering, Model Training & 
Evaluation
 Deep Learning: CNN, RessNet, MobileNet, ViT
 Generative AI: Prompt Engineering, RAG basics, LLM workflows, OpenAI APIs, LangChain (Basics) 
 Web Technologies: HTML, CSS, JavaScript (basics), Flask/FastAPI
 Tools: Git, GitHub
 Certificates  
ML Using Python – NIELIT | Web Technologies – SRKR College | Python (Basic) – HackerRank 
Programming in C++ – Cisco | Machine Learning – MLConnect | Modern AI – Cisco 
Achievements 

In [15]:
system_prompt = """
You are a helpful Retrieval-Augmented Generation assistant.

Answer the user's question using ONLY the provided document context.

Rules:
1. Do not invent information.
2. Do not use outside knowledge.
3. If the answer is not present in the context,
   say that the answer could not be found in the document.
4. Give a clear and concise answer.
"""

user_prompt = f"""
DOCUMENT CONTEXT:

{context}

QUESTION:

{question}

Answer the question using only the document context.
"""

response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ],
    temperature=0.2,
    max_tokens=800
)

answer = response.choices[0].message.content

print("ANSWER:")
print(answer)

ANSWER:
The programming languages mentioned in the resume are:

1. C
2. Python
3. Java
4. C++


In [16]:
print("RETRIEVED SOURCES")
print("=" * 50)

for index, metadata in enumerate(
    retrieved_metadatas,
    start=1
):
    print(
        f"Source {index}: "
        f"{metadata['source']} - "
        f"Page {metadata['page']}"
    )

RETRIEVED SOURCES
Source 1: AI_resume.pdf - Page 1
Source 2: AI_resume.pdf - Page 1
Source 3: AI_resume.pdf - Page 1
Source 4: AI_resume.pdf - Page 1
